# Regressão linear 

In [56]:
import pandas as pd
from pathlib import Path
import numpy as np
import statsmodels.formula.api as smf

Leitura do dataset.

In [57]:
caminho = Path(r'C:\Users\saulosgil\OneDrive\Documentos\consumo_alimentar_idososHosp\data\data_reg.xlsx')
df = pd.read_excel(caminho, engine='openpyxl', sheet_name= 'meta_kcal')

df.head()

,id,firstname,lastname,idade,sexo,NRS_2002,HAS,DMII,ICC,DRC,...,DoisGruposKcal,dias_internados,delta_csarf,delta_csavl,delta_hgs,delta_tug,delta_ts,delta_sppb,delta_barthel,cons_kcal_medio
0,35,Aderbal,Alves,99,Feminino,4,1,0,0,0,...,ProxMeta,21,-0.31,-0.74,0.0,1.99,15.55,-2,0,1276.746857
1,6,Alvarina,Magro,75,Feminino,3,1,1,1,0,...,ProxMeta,12,-0.43,-1.00,2.0,-5.10,0.00,2,0,1034.529083
2,37,Anisia,Rodrigues dos Santos,73,Feminino,4,1,1,0,0,...,ProxMeta,14,-0.19,-1.57,-2.0,0.63,1.94,-1,0,1048.831857
3,29,Antonia,Bigoli,80,Feminino,3,1,1,0,0,...,ProxMeta,6,-1.16,-1.31,10.0,0.79,-1.53,-2,-5,1340.373500
4,3,Aurora,Monici de Paula,81,Feminino,3,1,0,0,0,...,ProxMeta,7,-1.09,-0.31,1.0,-0.18,0.00,1,-5,1135.007143


Ajuste das variáveis para aplicar no modelo.

In [58]:
# Idade categorizada: '< 80 anos' ou '>= 80 anos' (idade ausente = NaN)
df['idade_cat'] = (pd.Series(np.where(df['idade'] < 80, '< 80 anos', '>= 80 anos'), index=df.index)
                   .where(df['idade'].notna()))

# Exposição em unidades de 100 kcal/dia: β = mudança no desfecho a cada +100 kcal/dia consumidas
df['kcal100'] = df['cons_kcal_medio'] / 100

# Regressão linear  

1. Desfechos
    - `AST VL`;
    - `AST RF`;                 
    - `HGS`;                  
    - `TUG`;        
    - `TST`; 
    - `SPPB`;        
    - `Barthel Index`;

In [59]:
ref = "kcal100"   # consumo energético médio, por 100 kcal/dia

desfechos = {
    'delta_csarf':      'CSARF (cm²)',
    'delta_csavl':      'CSAVL (cm²)',
    'delta_hgs':        'HGS (kgf)',
    'delta_tug':        'TUG (seconds)',
    'delta_ts':         'TST (seconds)',
    'delta_sppb':       'SPPB (a.u.)',
    'delta_barthel':    'Barthel (a.u.)',
}

resultados_reg = []

for var, rotulo in desfechos.items():

    formula = f"{var} ~ {ref}"

    # Modelo de regressão linear com erros-padrão robustos HC3
    m = smf.ols(formula, data=df).fit(cov_type='HC3')

    b      = m.params[ref]                # β por +100 kcal/dia
    lo, hi = m.conf_int().loc[ref]        # IC 95%
    p      = m.pvalues[ref]

    resultados_reg.append({
        'Outcome': rotulo,
        'β por 100 kcal (IC 95%)': f'{b:.3f} ({lo:.3f} a {hi:.3f})',
        'P': round(p, 3),
        'n': int(m.nobs),
        'R² ajustado': round(m.rsquared_adj, 2),
    })

tabela_reg = pd.DataFrame(resultados_reg)
tabela_reg

,Outcome,β por 100 kcal (IC 95%),P,n,R² ajustado
0,CSARF (cm²),0.031 (-0.011 a 0.072),0.150,54,0.03
1,CSAVL (cm²),0.110 (0.017 a 0.202),0.020,54,0.08
2,HGS (kgf),0.298 (-0.147 a 0.743),0.190,54,0.02
3,TUG (seconds),0.047 (-0.271 a 0.366),0.772,54,-0.02
4,TST (seconds),-0.101 (-0.926 a 0.724),0.810,54,-0.02
5,SPPB (a.u.),0.043 (-0.158 a 0.243),0.677,54,-0.02
6,Barthel (a.u.),-0.163 (-1.158 a 0.832),0.748,54,-0.02


# Regressão linear ajustada 

1. Desfechos
    - `AST VL`;
    - `AST RF`;                 
    - `HGS`;                  
    - `TUG`;        
    - `TST`; 
    - `SPPB`;        
    - `Barthel Index`;      

2. Covariáveis
    - `peso_adm`;
    - `sexo`;                 
    - `idade_cat`;                  
    - `NRS_2002`;        
    - `HAS`; 
    - `DMII`;        
    - `ICC`;   
    - `DRC`;        
    - `DPOC`;.      
    - `dias_internados`; 

In [60]:
ref = "kcal100"   # consumo energético médio, por 100 kcal/dia

desfechos = {
    'delta_csarf':      'CSARF (cm²)',
    'delta_csavl':      'CSAVL (cm²)',
    'delta_hgs':        'HGS (kgf)',
    'delta_tug':        'TUG (seconds)',
    'delta_ts':         'TST (seconds)',
    'delta_sppb':       'SPPB (a.u.)',
    'delta_barthel':    'Barthel (a.u.)',
}

# Covariáveis: peso_adm e dias_internados entram como contínuas (sem C());
# as demais são categóricas/binárias.
covariaveis = ['peso_adm', 'dias_internados',
               'C(sexo)', 'C(idade_cat)', 'C(NRS_2002)',
               'C(DMII)', 'C(HAS)', 'C(ICC)', 'C(DRC)', 'C(DPOC)']

resultados_reg = []

for var, rotulo in desfechos.items():
    formula = f"{var} ~ {ref} + {' + '.join(covariaveis)}"
    m = smf.ols(formula, data=df).fit(cov_type='HC3')   # EP robustos, adequados para N pequeno

    b      = m.params[ref]                # β por +100 kcal/dia
    lo, hi = m.conf_int().loc[ref]        # IC 95%
    p      = m.pvalues[ref]

    resultados_reg.append({
        'Outcome': rotulo,
        'β por 100 kcal (IC 95%)': f'{b:.3f} ({lo:.3f} a {hi:.3f})',
        'P': round(p, 3),
        'n': int(m.nobs),
        'nº parâmetros': len(m.params),
        'R² ajustado': round(m.rsquared_adj, 2),
    })

tabela_reg = pd.DataFrame(resultados_reg)
tabela_reg

,Outcome,β por 100 kcal (IC 95%),P,n,nº parâmetros,R² ajustado
0,CSARF (cm²),0.037 (-0.027 a 0.101),0.253,54,13,0.01
1,CSAVL (cm²),0.139 (0.024 a 0.254),0.018,54,13,0.13
2,HGS (kgf),0.324 (-0.450 a 1.098),0.412,54,13,-0.01
3,TUG (seconds),0.083 (-0.252 a 0.418),0.626,54,13,-0.07
4,TST (seconds),-0.069 (-1.269 a 1.130),0.910,54,13,-0.04
5,SPPB (a.u.),0.052 (-0.281 a 0.384),0.761,54,13,-0.02
6,Barthel (a.u.),-0.163 (-1.720 a 1.395),0.838,54,13,0.03
